## Task 3 part 2 - Modelling 1 (Baseline)

In [48]:
# load back in the data from Task3_01
import pandas as pd

DATA_DIR = "/home/ubuntu/data/frangieh"

pert_FC_selected = pd.read_pickle(f'{DATA_DIR}/task3_pert_FC_selected_50.pkl')
train_40 = pd.read_csv(f'{DATA_DIR}/task3_train_40.csv')['perturbation'].tolist()
test_10 = pd.read_csv(f'{DATA_DIR}/task3_test_10.csv')['perturbation'].tolist()

In [49]:
pert_FC_train = pert_FC_selected.loc[train_40].copy()
pert_FC_test = pert_FC_selected.loc[test_10].copy()

Now that we have the data sliced down to 50 perturbations and split into training and testing perturbations we will train our first model. This one is supposed to be "deliberately simplistic", as stated in the task description. 
Therefore we will call this the baseline model which for a test gene predicts its log2FC is just the average over all training perturbations in that condition. 

In [50]:
conditions = pert_FC_selected.index.get_level_values("condition").unique().tolist()

baseline_predictions  = {}
for cond in conditions:
    baseline_predictions[cond] = pert_FC_train.xs(cond, level = 'condition').mean(axis=0)

Now that we have computed the predictions for the test pertrubations as the average across all train perturbations in each conditions we will need to evaluate the performance of this very simplistic baseline model. For this we compute the correlation of the predicted log2FC vector and the actual log2FC vectors for each perturbation in each condition as well as the mean squared error as well as the standard deviations both across all conditions and for each condition separately. 

In [51]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

def evaluate_predictions_baseline(true_df, predictions_by_condition):
    # initialize records list
    records = []
    # iterate over testing data (two level index + FC vector)
    for (pert, cond), true_fc in true_df.iterrows():
        # predicted FC vectors from training are equal to average in each condition
        pred_fc = predictions_by_condition[cond]
        # compute pearson and spearman correlation
        pearson_r, pearson_p = pearsonr(true_fc, pred_fc)
        spearman_r, spearman_p = spearmanr(true_fc, pred_fc)
        # compute mse
        mse = np.mean((true_fc - pred_fc) ** 2)
        # append values to record list 
        records.append({
            "perturbation": pert,
            "condition": cond,
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_r": spearman_r,
            "spearman_p": spearman_p,
            "mse": mse,
        })
    return pd.DataFrame(records)

In [53]:
# per (perturbation, condition) scores on the test perturbations
baseline_eval = evaluate_predictions_baseline(pert_FC_test, baseline_predictions)

metrics = ["pearson_r", "spearman_r", "mse"]

# per-condition mean and std of corrleation and mse values (n=10) for biological interpretation
per_condition_baseline = baseline_eval.groupby("condition")[metrics].agg(["mean", "std"])

# pooled mean and std across all test pairs (n=30)
overall_baseline = baseline_eval[metrics].agg(["mean", "std"])

baseline_eval

,perturbation,condition,pearson_r,pearson_p,spearman_r,spearman_p,mse
0,HLA-F,Control,0.705177,0.000000e+00,0.331474,1.701774e-52,0.001883
1,HLA-F,IFNγ,0.791001,0.000000e+00,0.465710,3.264618e-108,0.001332
2,HLA-F,Co-culture,0.281587,9.072716e-38,0.333356,4.151949e-53,0.002481
3,CTSB,Control,0.814113,0.000000e+00,0.470140,1.610456e-110,0.001233
4,CTSB,IFNγ,0.895164,0.000000e+00,0.478541,5.461476e-115,0.000745
5,CTSB,Co-culture,0.867694,0.000000e+00,0.313631,6.748609e-47,0.001219
6,LCP1,Control,0.564849,0.000000e+00,0.471013,5.601197e-111,0.002829
7,LCP1,IFNγ,0.915148,0.000000e+00,0.483833,7.200037e-118,0.000870
8,LCP1,Co-culture,0.904633,0.000000e+00,0.331799,1.334453e-52,0.001194
9,LRPAP1,Control,0.851769,0.000000e+00,0.458810,1.097154e-104,0.001416


In [54]:
overall_baseline

,pearson_r,spearman_r,mse
mean,0.797456,0.405116,0.001465
std,0.174996,0.075286,0.000565


The results show that pearson correlation is high but spearman correlation is much lower, indicating that there are certain genes in each FC vector that have very dominant values that are flattened in the rank based computation. This is most likely the perturbed gene itself whichs expression crashes due to the KO. This is why the spearman correlation value is especially useful here to assess the overall performance of the model. This shows that the model's predictions are not totally random but also not very accurate and serve as a good staring point to reffer to when evaluating the other models.

In [55]:
per_condition_baseline

pearson_r           spearman_r                 mse          
                mean       std       mean       std      mean       std
condition                                                              
Co-culture  0.731887  0.274801   0.327853  0.025467  0.001501  0.000603
Control     0.802527  0.099753   0.444215  0.063247  0.001654  0.000568
IFNγ        0.857955  0.065856   0.443282  0.060536  0.001239  0.000495

Here we can see that the results are worst for the Co-culture condition and by far the most variable in the pearson correlation. When looking at the baseline_eval table we can see that this variability is caused by SERPINA3 and HLA-F which score only a correlation value of 0.17 and 0.28 respectively. On the other hand the IFN treated condition is the most consistent in this regard which lines up with the fact that IFNγ treatment triggers a very distinct and broad response programm that drives many genes in the same direction and thus an average mirrors this response more closely than for the Co-culture condition which seems to produce a much more heterogeneous response. 

The next models we train should aim to avoid any condition specific differences in performance and to elevate especially the spearman correlation value. 